# Sorting and Safe Reporting Workloads


```{contents}
:local:
:depth: 2
```


```{index} Maintain a Sorted Prefix While Preserving Every Record
```



```{index} Maintain a Sorted Prefix While Preserving Every Record
```

## Maintain a Sorted Prefix While Preserving Every Record



Sorting has at least two obligations: output order must satisfy the chosen comparison policy, and output must contain exactly the input occurrences. Producing 1, 2, 3 from input 3, 1, 2 looks right, but so would returning a fixed 1, 2, 3 regardless of input. Tests need to challenge preservation as well as ordering.

Insertion sort maintains a sorted prefix. The next item is saved outside the moving region. Larger prefix items shift one position right until the saved item fits. Assigning the saved item back closes the hole. This explains why writing over the next item before saving it can lose data even though the remaining positions appear ordered.

| Input prefix and next item | Work | Resulting prefix |
| --- | --- | --- |
| [4], next 2 | Save 2, shift 4, insert 2 | [2, 4] |
| [2, 4], next 3 | Save 3, shift 4, insert 3 | [2, 3, 4] |

Before each outer iteration, the processed prefix is sorted and is a permutation of the corresponding original prefix. The inner shifts preserve all saved occurrences while opening the correct position. When the prefix includes the full array, both obligations hold. This is a reasoning structure, not simply a description of what a successful example happened to print.

Equal keys reveal another contract. If the inner loop moves only items strictly greater than the saved key, equal-key items retain their previous relative order. This is stability. Moving equal items as well can change that order. Use labeled records, such as two invoices with the same total but different IDs, to see the distinction; identical integer values cannot show which occurrence moved.

For related explanation, see the further reading below.[^development-reading]


```{index} Preparation, Mutation, and the Report Caller
```



```{index} Preparation, Mutation, and the Report Caller
```

## Preparation, Mutation, and the Report Caller



Sorting an array in place changes the order seen through its aliases. If arrival order is a business record, sort an independently ordered outer copy. That preserves original positions but still shares references to mutable objects. State whether the report needs only an order snapshot or a deeper data snapshot.

Guided check: sort quantities 4, 2, 3 and record both comparisons and shifts. Then use already sorted 2, 3, 4. Explain why the amount of work differs even though both inputs have length three. Finally, test all equal values and reverse order, checking that each result preserves occurrence counts. A sorting algorithm's name is not a substitute for these contracts.


```{index} Basic Sorting with Insertion Sort
```

<span id="basic-sorting-with-insertion-sort"></span>


A payroll report may need employees in name order or invoices in amount order. A sorting algorithm must preserve every occurrence while arranging elements according to a comparison rule. Calling a library sort is useful, but implementing one small sort lets you see the work behind that operation.

Insertion sort is the core teaching algorithm in this chapter. It keeps a sorted prefix and inserts one new value into that prefix. Advanced sorting implementations and detailed comparisons remain in Part VI.


```{index} Insert into a Sorted Prefix
```

## Insert into a Sorted Prefix




```{index} Insert into a Sorted Prefix
```


At the start of iteration i, positions zero through i-1 are sorted and contain the same occurrences as before. Save values[i] before shifting anything. Move larger prefix values one position to the right, then store the saved value in the gap.

For [30,10,20], insert 10 into [30] to obtain [10,30,20]. Then save 20, shift 30, and place 20 between 10 and 30. The array is never extended or shortened. The saved current value prevents shifts from overwriting the item being inserted.

| Pass | Sorted prefix afterward | Remaining values |
| --- | --- | --- |
| Start | 30 | 10, 20 |
| Insert 10 | 10, 30 | 20 |
| Insert 20 | 10, 20, 30 | None |

Use a strict greater-than comparison when shifting. Equal keys stay in their original relative order, which makes this implementation stable.


In [1]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20, 10};
long comparisons = BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(", ", amounts));
Console.WriteLine($"Comparisons: {comparisons}");

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10, 10, 20, 30
Comparisons: 6


```{index} Sorting Objects Uses an Ordering Policy
```

## Sorting Objects Uses an Ordering Policy




```{index} Sorting Objects Uses an Ordering Policy
```


The comparer returns a negative value, zero, or a positive value. These mean before, tied, or after; callers should test the sign rather than assume the magnitude is exactly one. Define ordering separately from object identity and business equality.

The invoice comparer orders by amount only. Two invoices with the same amount are tied even though they have different identifiers. Stable sorting retains their arrival order. If the business requires amount then ID, include that second key in the comparer rather than relying on an accidental order.


In [1]:
using System;
using System.Collections.Generic;

Invoice[] invoices = {new Invoice("A", 20m), new Invoice("B", 10m), new Invoice("C", 20m)};
BasicSort.InsertionSort(invoices, new AmountComparer());
foreach (Invoice invoice in invoices) Console.WriteLine($"{invoice.Id}: {invoice.Amount:F2}");

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}

public class Invoice
{
    public string Id { get; }
    public decimal Amount { get; }
    public Invoice(string id, decimal amount) { Id=id; Amount=amount; }
}
public class AmountComparer : IComparer<Invoice>
{
    public int Compare(Invoice? left, Invoice? right)
    {
        if (left is null || right is null) throw new ArgumentException("Invoices required.");
        return left.Amount.CompareTo(right.Amount);
    }
}


B: 10.00
A: 20.00
C: 20.00


```{index} Count Work for Different Inputs
```

## Count Work for Different Inputs




```{index} Count Work for Different Inputs
```


On already sorted data, each new item needs one failed shift comparison, for n-1 comparisons when n is positive. On distinct reverse-sorted data, item i crosses the entire prefix: 1+2+...+(n-1) = n(n-1)/2 comparisons. Therefore the implementation has linear best-case and quadratic worst-case comparison work.

The array is sorted in place with constant auxiliary element storage. Shifting still performs writes, so comparisons are not the only possible cost model. Use the counter to test a specific comparison claim; do not confuse the counter with wall-clock time.


In [1]:
using System;
using System.Collections.Generic;

foreach (int[] input in new[] {Array.Empty<int>(), new[]{1}, new[]{1,2,3,4}, new[]{4,3,2,1}})
{
    long count = BasicSort.InsertionSort(input, Comparer<int>.Default);
    Console.WriteLine($"n={input.Length}, comparisons={count}, result={string.Join(",",input)}");
}

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


n=0, comparisons=0, result=
n=1, comparisons=0, result=1
n=4, comparisons=3, result=1,2,3,4
n=4, comparisons=6, result=1,2,3,4


```{index} Verify Order and Preservation
```

## Verify Order and Preservation




```{index} Verify Order and Preservation
```


A sorted result alone is insufficient: a broken implementation could discard a value and still look ordered. Check length and occurrence preservation as well as ordering. Use a separately sorted copy as a small reference, and include duplicates, negative values, empty input, a singleton, and already sorted data.

In application code, normally use the library operation unless a requirement justifies a custom algorithm. Array.Sort mutates its array and does not promise stable ties. Copy before sorting when callers need the original sequence. Part VI investigates additional sorting algorithms and their tradeoffs.


```{index} Practice: Preserve the Saved Item
```

## Practice: Preserve the Saved Item




```{index} Practice: Preserve the Saved Item
```


Complete the final insertion after shifting larger values. Do not lose the saved current item.


In [ ]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20};
BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(",", amounts));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            // Your code starts here.
            /* TODO: store current in the gap. */
            // Your code ends here.
        }
        return comparisons;
    }
}


In [1]:
using System;
using System.Collections.Generic;

int[] amounts = {30, 10, 20};
BasicSort.InsertionSort(amounts, Comparer<int>.Default);
Console.WriteLine(string.Join(",", amounts));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10,20,30


```{index} Library Operations and Clear Comparers
```

## Library Operations and Clear Comparers




```{index} Library Operations and Clear Comparers
```


Array.Sort and List.Sort are ordinary application tools. Supply the ordering policy when sorting custom types, and preserve input if mutation is forbidden. Comparers should be consistent: a value compares equal to itself, reversing operands reverses the ordering sign, and the ordering should not form contradictory cycles.

For displayed text, choose an ordering suitable for users. For symbolic keys in the core examples, ordinal comparison makes the rule explicit. A search must use the same ordering that created its sorted input.


In [1]:
using System;
using System.Collections.Generic;

string[] ids = {"T-200", "T-010", "T-100"};
Array.Sort(ids, StringComparer.Ordinal);
Console.WriteLine(string.Join(",", ids));
Console.WriteLine(Array.BinarySearch(ids, "T-100", StringComparer.Ordinal));


T-010,T-100,T-200
1


```{index} Design a Boundary Matrix
```

## Design a Boundary Matrix




```{index} Design a Boundary Matrix
```


A small check matrix can include empty input, one element, a missing key before or after the range, duplicates, all-equal values, sorted input, and reverse input. For object sorting, check a stable-tie example against the required policy.

Keep each result contract explicit. A first-match linear search, any-match binary search, and “all matching invoices” query are different tasks. Do not claim one implementation is faster by giving it less work or a weaker required answer.

In the connected lab, prepare a ticket report, sort it, search it, and check preserved input and duplicate counts. In homework, transfer those skills to rosters, amounts, and differently ordered records. The next chapters use ordered operations inside trees and priority structures.


```{index} Practice: Choose a Safe Reporting Snapshot
```

## Practice: Choose a Safe Reporting Snapshot




```{index} Practice: Choose a Safe Reporting Snapshot
```


Create an independently ordered array before sorting; the original arrival order must remain unchanged. Expected output: `10,40,40`, and the check line must not throw.


In [ ]:
using System;
using System.Collections.Generic;

int[] original = {40, 10, 40};
int[] sorted = original /* TODO: copy the outer array */;
BasicSort.InsertionSort(sorted, Comparer<int>.Default);
if (original[0] != 40 || sorted.Length != original.Length) throw new Exception("Snapshot failed.");
Console.WriteLine(string.Join(",", sorted));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


In [1]:
using System;
using System.Collections.Generic;

int[] original = {40, 10, 40};
int[] sorted = (int[])original.Clone();
BasicSort.InsertionSort(sorted, Comparer<int>.Default);
if (original[0] != 40 || sorted.Length != original.Length) throw new Exception("Snapshot failed.");
Console.WriteLine(string.Join(",", sorted));

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


10,40,40


```{index} Search and Sort Workloads
```

## Compare Reporting and Query Workloads



An algorithm choice should answer a business requirement, not just win one small timing demonstration. A daily report, a one-time lookup, and an interactive index have different workloads even when all use the same records.

Combine searching, ordering, copying, and validation into one task. First say what may change and what the caller must receive. Then choose an operation and explain both its preparation cost and its repeated-use cost.


```{index} Preserve Arrival Order While Preparing a Report
```

## Preserve Arrival Order While Preparing a Report




```{index} Preserve Arrival Order While Preparing a Report
```


Sort a copy when the original sequence records arrival order. An array clone is shallow: it copies element values or object references, not every reachable object. Sorting the new array changes its order without changing the original array's order.

This distinction extends Chapter 11's ownership discussion. A report snapshot is not an independent copy of mutable invoice objects. Decide whether records themselves may change while the report is used.


In [1]:
using System;
using System.Collections.Generic;

int[] arrived = {300, 100, 200, 100};
int[] report = (int[])arrived.Clone();
BasicSort.InsertionSort(report, Comparer<int>.Default);
Console.WriteLine("Arrival: " + string.Join(",", arrived));
Console.WriteLine("Report: " + string.Join(",", report));
Console.WriteLine(BasicSearch.BinaryAny(report, 200, Comparer<int>.Default));

public static class BasicSearch
{
    public static int LinearFirst<T>(T[] values, T target, IEqualityComparer<T> equality)
    {
        for (int i = 0; i < values.Length; i++)
            if (equality.Equals(values[i], target)) return i;
        return -1;
    }
    public static int BinaryAny<T>(T[] sorted, T target, IComparer<T> comparer)
    {
        int low = 0, high = sorted.Length;
        while (low < high)
        {
            int middle = low + (high - low) / 2;
            int comparison = comparer.Compare(sorted[middle], target);
            if (comparison == 0) return middle;
            if (comparison < 0) low = middle + 1;
            else high = middle;
        }
        return -1;
    }
}

public static class BasicSort
{
    public static long InsertionSort<T>(T[] values, IComparer<T> comparer)
    {
        long comparisons = 0;
        for (int i = 1; i < values.Length; i++)
        {
            T current = values[i];
            int position = i;
            while (position > 0)
            {
                comparisons++;
                if (comparer.Compare(values[position - 1], current) <= 0) break;
                values[position] = values[position - 1];
                position--;
            }
            values[position] = current;
        }
        return comparisons;
    }
}


Arrival: 300,100,200,100
Report: 100,100,200,300
2


```{index} Include Preparation and Updates
```

## Include Preparation and Updates




```{index} Include Preparation and Updates
```


For q searches over n records, repeated linear scans can require qn equality checks. A prepared sorted array needs sorting work plus q logarithmic searches. In this chapter's insertion-sort implementation, worst-case preparation is quadratic; a standard efficient comparison sort has a different preparation bound. Do not assign a library sort's cost to the teaching insertion sort.

The benefit also depends on updates. A static end-of-day report can be prepared once. An interactive record set may require maintaining order or rebuilding a snapshot. Later hash collections provide another option for exact-key lookup; trees support ordered access. There is no single structure that is best for every operation.


```{rubric} Footnotes
```


[^1602_basic_sorting-1]: Stability concerns the relative order of equal comparison keys. It does not mean every object is deeply copied or that objects with the same key have the same identity.


[^1603_workloads-1]: Measured timings depend on the runtime and inputs. Operation counts and stated assumptions support a growth argument; one fast run does not establish a general complexity bound.


[^development-reading]: Further reading: [OpenDSA, Search in Sorted Arrays](https://opendsa.cs.vt.edu/OpenDSA/Books/Catalog/html/SortedSearch.html). The worked business scenarios and guided reasoning here are original teaching examples.
